In [1]:
pip install onnx onnxruntime opencv-python-headless numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 41.7 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import subprocess
import sys
from pathlib import Path
import cv2
import numpy as np
import onnx
from onnxruntime.quantization import (
   CalibrationDataReader,
   CalibrationMethod,
   QuantFormat,
   QuantType,
   quantize_static,
)

class YOLOv12CalibrationDataReader(CalibrationDataReader):
   """
   Letterbox C++ for reading data.
   Alligned with src/face_detection_yolov12.py for the same data distribution.
   """
   def __init__(self, image_dir: str, input_name: str = "images", input_size=(640, 640), max_images: int = 150):
       self.image_dir = Path(image_dir)
       self.input_name = input_name
       self.input_width, self.input_height = input_size
       valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
       self.image_paths = [
           p for p in self.image_dir.iterdir() if p.suffix.lower() in valid_exts
       ][:max_images]
       if not self.image_paths:
           raise ValueError(f"Không tìm thấy ảnh nào trong thư mục: {self.image_dir}")
       print(f"[*] Đã nạp {len(self.image_paths)} ảnh hiệu chuẩn từ '{self.image_dir}'.")
       self._iterator = None
   def _preprocess_letterbox(self, image: np.ndarray) -> np.ndarray:
       img_h, img_w = image.shape[:2]
       scale = min(self.input_width / img_w, self.input_height / img_h)
       new_w, new_h = int(round(img_w * scale)), int(round(img_h * scale))
       pad_w = (self.input_width - new_w) / 2.0
       pad_h = (self.input_height - new_h) / 2.0
       resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
       top, bottom = int(round(pad_h - 0.1)), int(round(pad_h + 0.1))
       left, right = int(round(pad_w - 0.1)), int(round(pad_w + 0.1))
       padded = cv2.copyMakeBorder(
           resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114)
       )
       blob = cv2.dnn.blobFromImage(
           padded, 1.0 / 255.0, (self.input_width, self.input_height), swapRB=True, crop=False
       )
       return blob
   def get_next(self):
       if self._iterator is None:
           batches = []
           for img_path in self.image_paths:
               img = cv2.imread(str(img_path))
               if img is not None:
                   blob = self._preprocess_letterbox(img)
                   batches.append({self.input_name: blob})
           self._iterator = iter(batches)
       return next(self._iterator, None)

def get_head_nodes_to_exclude(onnx_model_path: str) -> list[str]:
   """
   Automatically find calculating nodes at Detection Head
   to keep the FP32 accuracy, avoid slip bounding box.
   """
   model = onnx.load(onnx_model_path)
   exclude_nodes = []
   sensitive_ops = {"Sub", "Add", "Mul", "Div", "Sigmoid", "Softmax", "Pow"}
   for node in model.graph.node:
       if node.op_type in sensitive_ops:
           exclude_nodes.append(node.name)
   print(f"[*] Locked {len(exclude_nodes)} crucial mathematical nodes at FP32.")
   return exclude_nodes

def quantize_yolov12_model(fp32_model_path: str, calib_dir: str):
   fp32_path = Path(fp32_model_path)
   int8_path = fp32_path.with_name(f"{fp32_path.stem}-int8.onnx")
   print(f"\n[1/2] Loading and check graphs for {fp32_path.name}...")
   model = onnx.load(str(fp32_path))

   try:
       inferred_model = onnx.shape_inference.infer_shapes(model)
       temp_path = fp32_path.with_name(f"{fp32_path.stem}-inferred.onnx")
       onnx.save(inferred_model, str(temp_path))
       target_input_model = str(temp_path)
   except Exception as e:
       print(f"[*] Warning shape inference: {e}. Using root files.")
       target_input_model = str(fp32_path)
       temp_path = None

   model_for_input = onnx.load(target_input_model)
   input_name = model_for_input.graph.input[0].name
   nodes_to_exclude = get_head_nodes_to_exclude(target_input_model)
    
   print(f"[2/2] Executing Calibration and INT8 Quantization...")
   calib_reader = YOLOv12CalibrationDataReader(calib_dir, input_name=input_name)
   quantize_static(
       model_input=target_input_model,
       model_output=str(int8_path),
       calibration_data_reader=calib_reader,
       quant_format=QuantFormat.QDQ,
       calibrate_method=CalibrationMethod.MinMax,
       activation_type=QuantType.QUInt8,
       weight_type=QuantType.QInt8,
       op_types_to_quantize=["Conv", "MatMul"],
       nodes_to_exclude=nodes_to_exclude,
       per_channel=True,
       reduce_range=False,
   )
   if temp_path and Path(temp_path).exists():
       Path(temp_path).unlink()
   fp32_size = fp32_path.stat().st_size / (1024 * 1024)
   int8_size = int8_path.stat().st_size / (1024 * 1024)
   print(f"[3/3] Done! Saved at: {int8_path}")
   print(f"      Memory is saved from {fp32_size:.2f} MB to {int8_size:.2f} MB (Saved {(1 - int8_size/fp32_size)*100:.1f}%)")

if __name__ == "__main__":
   CALIB_IMAGES_DIR = "/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba"
   models_to_quantize = [
       "/kaggle/input/models/revdra/yolov12-face-onnx/onnx/default/1/yolov12n-face.onnx",
       "/kaggle/input/models/revdra/yolov12-face-onnx/onnx/default/1/yolov12s-face.onnx",
       "/kaggle/input/models/revdra/yolov12-face-onnx/onnx/default/1/yolov12m-face.onnx",
       "/kaggle/input/models/revdra/yolov12-face-onnx/onnx/default/1/yolov12l-face.onnx"
   ]
   for model_path in models_to_quantize:
       if Path(model_path).exists():
           local_model_name = Path(model_path).name
           local_model_path = Path(f"/kaggle/working/{local_model_name}")
           import shutil
           shutil.copy(model_path, local_model_path)
           print(f"[*] Transforming: {local_model_path}")
           quantize_yolov12_model(str(local_model_path), CALIB_IMAGES_DIR)
       else:
           print(f"[!] File unfound: {model_path}")

[*] Transforming: /kaggle/working/yolov12n-face.onnx

[1/2] Loading and check graphs for yolov12n-face.onnx...
[*] Locked 253 crucial mathematical nodes at FP32.
[2/2] Executing Calibration and INT8 Quantization...


[*] Đã nạp 150 ảnh hiệu chuẩn từ '/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba'.


[3/3] Done! Saved at: /kaggle/working/yolov12n-face-int8.onnx
      Memory is saved from 10.47 MB to 3.62 MB (Saved 65.5%)
[*] Transforming: /kaggle/working/yolov12s-face.onnx

[1/2] Loading and check graphs for yolov12s-face.onnx...
[*] Locked 253 crucial mathematical nodes at FP32.
[2/2] Executing Calibration and INT8 Quantization...


[*] Đã nạp 150 ảnh hiệu chuẩn từ '/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba'.


[3/3] Done! Saved at: /kaggle/working/yolov12s-face-int8.onnx
      Memory is saved from 35.93 MB to 10.15 MB (Saved 71.8%)
[*] Transforming: /kaggle/working/yolov12m-face.onnx

[1/2] Loading and check graphs for yolov12m-face.onnx...
[*] Locked 275 crucial mathematical nodes at FP32.
[2/2] Executing Calibration and INT8 Quantization...


[*] Đã nạp 150 ảnh hiệu chuẩn từ '/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba'.


[3/3] Done! Saved at: /kaggle/working/yolov12m-face-int8.onnx
      Memory is saved from 77.45 MB to 20.73 MB (Saved 73.2%)
[*] Transforming: /kaggle/working/yolov12l-face.onnx

[1/2] Loading and check graphs for yolov12l-face.onnx...
[*] Locked 455 crucial mathematical nodes at FP32.
[2/2] Executing Calibration and INT8 Quantization...


[*] Đã nạp 150 ảnh hiệu chuẩn từ '/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba'.


[3/3] Done! Saved at: /kaggle/working/yolov12l-face-int8.onnx
      Memory is saved from 101.82 MB to 27.74 MB (Saved 72.8%)


In [3]:
import cv2
import numpy as np
import onnxruntime as ort
from pathlib import Path
class YOLOv12FaceDetector:
   """
   Class for detecting faces using YOLOv12 at ONNX (Pure ONNX Runtime + OpenCV).
   """
   def __init__(self, model_path: str, conf_threshold: float = 0.32, iou_threshold: float = 0.5):
       self.model_path = Path(model_path)
       if not self.model_path.exists():
           raise FileNotFoundError(f"Không tìm thấy mô hình tại: {self.model_path}")
       self.conf_threshold = conf_threshold
       self.iou_threshold = iou_threshold
       # ONNX Runtime Session Options
       options = ort.SessionOptions()
       options.intra_op_num_threads = 4
       options.inter_op_num_threads = 1
       options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
       # Provider Search (CUDA if yes, else CPU)
       available_providers = ort.get_available_providers()
       providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if 'CUDAExecutionProvider' in available_providers else ['CPUExecutionProvider']
       print(f"[*] Loading model: {self.model_path.name}")
       print(f"[*] ONNX Providers: {providers}")
       self.session = ort.InferenceSession(str(self.model_path), options, providers=providers)
       # Get input size of the model
       input_meta = self.session.get_inputs()[0]
       self.input_name = input_meta.name
       input_shape = input_meta.shape
       # Default YOLOv12 uses 640x640
       self.input_height = input_shape[2] if isinstance(input_shape[2], int) else 640
       self.input_width = input_shape[3] if isinstance(input_shape[3], int) else 640
   def _preprocess(self, img: np.ndarray):
       """Leterbox C++"""
       img_h, img_w = img.shape[:2]
       scale = min(self.input_width / img_w, self.input_height / img_h)
       new_w, new_h = int(round(img_w * scale)), int(round(img_h * scale))
       pad_w = (self.input_width - new_w) / 2.0
       pad_h = (self.input_height - new_h) / 2.0
       resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
       top, bottom = int(round(pad_h - 0.1)), int(round(pad_h + 0.1))
       left, right = int(round(pad_w - 0.1)), int(round(pad_w + 0.1))
       padded = cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114))
       blob = cv2.dnn.blobFromImage(padded, 1.0 / 255.0, (self.input_width, self.input_height), swapRB=True, crop=False)
       return blob, scale, (left, top)
   def detect_faces(self, img: np.ndarray):
       """Inference and return results."""
       original_h, original_w = img.shape[:2]
       blob, scale, (pad_x, pad_y) = self._preprocess(img)

       outputs = self.session.run(None, {self.input_name: blob})
       raw_output = outputs[0]

       if raw_output.ndim == 3:
           raw_output = raw_output.transpose(0, 2, 1)
       predictions = raw_output[0]
       boxes = []
       confidences = []
       for pred in predictions:
           confidence = float(pred[4])
           if confidence >= self.conf_threshold:
               cx, cy, w, h = pred[0], pred[1], pred[2], pred[3]

               x1 = cx - w / 2.0
               y1 = cy - h / 2.0
               x2 = cx + w / 2.0
               y2 = cy + h / 2.0
    
               x1 = (x1 - pad_x) / scale
               y1 = (y1 - pad_y) / scale
               x2 = (x2 - pad_x) / scale
               y2 = (y2 - pad_y) / scale

               x1 = max(0.0, min(x1, original_w))
               y1 = max(0.0, min(y1, original_h))
               x2 = max(0.0, min(x2, original_w))
               y2 = max(0.0, min(y2, original_h))

               box_w = x2 - x1
               box_h = y2 - y1
               boxes.append([int(x1), int(y1), int(box_w), int(box_h)])
               confidences.append(confidence)
       # Non-Maximum Suppression (NMS)
       indices = cv2.dnn.NMSBoxes(boxes, confidences, self.conf_threshold, self.iou_threshold)
       results = []
       if len(indices) > 0:
           for i in indices.flatten():
               x, y, w, h = boxes[i]
               results.append({
                   "box": [x, y, x + w, y + h],
                   "confidence": float(confidences[i])
               })
       return results

In [4]:
import time
import cv2
test_img = cv2.imread("/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba/000001.jpg")
fp32 = ["/kaggle/working/yolov12n-face.onnx",
       "/kaggle/working/yolov12s-face.onnx",
       "/kaggle/working/yolov12m-face.onnx",
       "/kaggle/working/yolov12l-face.onnx"]
int8 = ["/kaggle/working/yolov12n-face-int8.onnx",
       "/kaggle/working/yolov12s-face-int8.onnx",
       "/kaggle/working/yolov12m-face-int8.onnx",
       "/kaggle/working/yolov12l-face-int8.onnx"]
for i in range(4):
    det_fp32 = YOLOv12FaceDetector(fp32[i])
    det_int8 = YOLOv12FaceDetector(int8[i])
    # Warm-up
    for _ in range(5):
       det_fp32.detect_faces(test_img)
       det_int8.detect_faces(test_img)
    # Benchmark FP32
    t0 = time.perf_counter()
    for _ in range(50):
       boxes_fp32 = det_fp32.detect_faces(test_img)
    t_fp32 = (time.perf_counter() - t0) / 50 * 1000
    # Benchmark INT8
    t0 = time.perf_counter()
    for _ in range(50):
       boxes_int8 = det_int8.detect_faces(test_img)
    t_int8 = (time.perf_counter() - t0) / 50 * 1000
    print(f"FP32: {t_fp32:.2f} ms | Detected: {len(boxes_fp32)} face(s)")
    print(f"INT8: {t_int8:.2f} ms | Detected: {len(boxes_int8)} face(s)")
    print(f"=> Speedup: {t_fp32 / t_int8:.2f}x")

[*] Loading model: yolov12n-face.onnx
[*] ONNX Providers: ['CPUExecutionProvider']
[*] Loading model: yolov12n-face-int8.onnx
[*] ONNX Providers: ['CPUExecutionProvider']
FP32: 113.33 ms | Detected: 1 face(s)
INT8: 129.96 ms | Detected: 1 face(s)
=> Speedup: 0.87x
[*] Loading model: yolov12s-face.onnx
[*] ONNX Providers: ['CPUExecutionProvider']
[*] Loading model: yolov12s-face-int8.onnx
[*] ONNX Providers: ['CPUExecutionProvider']
FP32: 254.79 ms | Detected: 1 face(s)
INT8: 264.81 ms | Detected: 1 face(s)
=> Speedup: 0.96x
[*] Loading model: yolov12m-face.onnx
[*] ONNX Providers: ['CPUExecutionProvider']
[*] Loading model: yolov12m-face-int8.onnx
[*] ONNX Providers: ['CPUExecutionProvider']
FP32: 638.70 ms | Detected: 1 face(s)
INT8: 586.05 ms | Detected: 1 face(s)
=> Speedup: 1.09x
[*] Loading model: yolov12l-face.onnx
[*] ONNX Providers: ['CPUExecutionProvider']
[*] Loading model: yolov12l-face-int8.onnx
[*] ONNX Providers: ['CPUExecutionProvider']
FP32: 904.84 ms | Detected: 1 face

In [5]:
import cv2
import numpy as np
from pathlib import Path
def calculate_iou(box1, box2):
   """
   Calculate Intersection over Union metrics between 2 bounding boxes [x1, y1, x2, y2].
   """
   x1 = max(box1[0], box2[0])
   y1 = max(box1[1], box2[1])
   x2 = min(box1[2], box2[2])
   y2 = min(box1[3], box2[3])
   inter_area = max(0, x2 - x1) * max(0, y2 - y1)
   if inter_area == 0:
       return 0.0
   box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
   box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
   union_area = box1_area + box2_area - inter_area
   return inter_area / union_area if union_area > 0 else 0.0
def compare_fp32_vs_int8(fp32_model_path: str, int8_model_path: str, image_path: str):
   """
   Compare results between FP32 & INT8
   """
   detector_fp32 = YOLOv12FaceDetector(fp32_model_path)
   detector_int8 = YOLOv12FaceDetector(int8_model_path)
   img = cv2.imread(str(image_path))
   if img is None:
       print(f"[!] Can't read files at: {image_path}")
       return
   res_fp32 = detector_fp32.detect_faces(img)
   res_int8 = detector_int8.detect_faces(img)
   print(f"\n" + "="*50)
   print(f" ACURRACY TEST: FP32 vs INT8")
   print(f"="*50)
   print(f" Tested Image : {Path(image_path).name}")
   print(f" FP32 : {len(res_fp32)} faces")
   print(f" INT8 : {len(res_int8)} faces")
   print("-" * 50)
   print("\nBounding Boxes (FP32):")
   for i, det in enumerate(res_fp32):
       print(f"  - Face [{i+1}] Box: {det['box']} | Conf: {det['confidence']*100:.2f}%")
   print("\nBounding Boxes (INT8):")
   for i, det in enumerate(res_int8):
       print(f"  - Face [{i+1}] Box: {det['box']} | Conf: {det['confidence']*100:.2f}%")

   if res_fp32 and res_int8:
       print("\nIoU Matching:")
       for idx_fp, d_fp in enumerate(res_fp32):
           best_iou = 0.0
           best_match_conf = 0.0
           best_box_int8 = None
           for d_in in res_int8:
               iou = calculate_iou(d_fp['box'], d_in['box'])
               if iou > best_iou:
                   best_iou = iou
                   best_match_conf = d_in['confidence']
                   best_box_int8 = d_in['box']
           print(f"  • FP32 Box {d_fp['box']} matches INT8 Box {best_box_int8}")
           print(f"    -> IoU: {best_iou:.4f}")
           print(f"    -> Confidence Mismatch: {abs(d_fp['confidence'] - best_match_conf)*100:.2f}%\n")
   else:
       print("\n[!] One of the models found nothing.")

if __name__ == "__main__":
    fp32 = ["/kaggle/working/yolov12n-face.onnx",
       "/kaggle/working/yolov12s-face.onnx",
       "/kaggle/working/yolov12m-face.onnx",
       "/kaggle/working/yolov12l-face.onnx"]
    int8 = ["/kaggle/working/yolov12n-face-int8.onnx",
       "/kaggle/working/yolov12s-face-int8.onnx",
       "/kaggle/working/yolov12m-face-int8.onnx",
       "/kaggle/working/yolov12l-face-int8.onnx"]
    TEST_IMAGE = "/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba/000002.jpg" # Thay bằng ảnh bất kỳ của bạn
    for i in range(4):
        compare_fp32_vs_int8(fp32[i], int8[i], TEST_IMAGE)

[*] Loading model: yolov12n-face.onnx
[*] ONNX Providers: ['CPUExecutionProvider']
[*] Loading model: yolov12n-face-int8.onnx
[*] ONNX Providers: ['CPUExecutionProvider']

 ACURRACY TEST: FP32 vs INT8
 Tested Image : 000002.jpg
 FP32 : 1 faces
 INT8 : 1 faces
--------------------------------------------------

Bounding Boxes (FP32):
  - Face [1] Box: [54, 63, 145, 188] | Conf: 91.44%

Bounding Boxes (INT8):
  - Face [1] Box: [53, 63, 145, 187] | Conf: 91.53%

IoU Matching:
  • FP32 Box [54, 63, 145, 188] matches INT8 Box [53, 63, 145, 187]
    -> IoU: 0.9813
    -> Confidence Mismatch: 0.09%

[*] Loading model: yolov12s-face.onnx
[*] ONNX Providers: ['CPUExecutionProvider']
[*] Loading model: yolov12s-face-int8.onnx
[*] ONNX Providers: ['CPUExecutionProvider']

 ACURRACY TEST: FP32 vs INT8
 Tested Image : 000002.jpg
 FP32 : 1 faces
 INT8 : 1 faces
--------------------------------------------------

Bounding Boxes (FP32):
  - Face [1] Box: [54, 62, 144, 188] | Conf: 89.70%

Bounding Box